# Chapter 14 — Learning to Write Like Shakespeare: Long Short-Term Memory

**Nama:** Omar Al Farizy Hutagalung  
**NIM:** 101032300034  
**Mata Kuliah:** Deep Learning

## Tujuan pembelajaran
Bab ini membahas *character language modeling*, truncated backpropagation through time (BPTT), vanishing dan exploding gradients, contoh backpropagation pada RNN, serta long short-term memory (LSTM).

Notebook ini adalah pendamping belajar yang disederhanakan, bukan salinan kode buku secara verbatim. Dataset Shakespeare asli tidak disertakan.

## 1. Character language modeling
Model memprediksi karakter berikutnya berdasarkan karakter sebelumnya. Vocabulary dibuat dari karakter unik, bukan kata. Kita gunakan teks kecil agar contoh dapat berjalan tanpa mengunduh dataset.

In [ ]:
import numpy as np

text = "to be or not to be"
vocab = sorted(set(text))
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
encoded = np.array([char_to_idx[ch] for ch in text], dtype=int)

print("Teks:", repr(text))
print("Vocabulary:", vocab)
print("Pemetaan karakter:", char_to_idx)
print("Indeks:", encoded)

### Membentuk pasangan input dan target
Setiap karakter dipasangkan dengan karakter setelahnya. Model belajar memperkirakan distribusi probabilitas karakter target dari karakter sebelumnya.

In [ ]:
inputs = encoded[:-1]
targets = encoded[1:]
for x_id, y_id in zip(inputs, targets):
    print(f"{idx_to_char[x_id]!r} -> {idx_to_char[y_id]!r}")

## 2. Truncated backpropagation through time
Pada RNN, hidden state diteruskan sepanjang urutan. Backpropagation melalui urutan yang sangat panjang membutuhkan memori dan komputasi besar. **Truncated BPTT** membatasi jumlah langkah waktu yang dilalui gradien. Panjang potongan menjadi hiperparameter; komprominya, model lebih sulit belajar ketergantungan yang jauh di masa lalu.

In [ ]:
def make_bptt_chunks(sequence, bptt):
    # Input dan target bergeser satu posisi; setiap chunk maksimal bptt langkah.
    chunks = []
    for start in range(0, len(sequence) - 1, bptt):
        end = min(start + bptt, len(sequence) - 1)
        x = sequence[start:end]
        y = sequence[start + 1:end + 1]
        if len(x):
            chunks.append((x, y))
    return chunks

demo_sequence = np.arange(14)
for i, (x, y) in enumerate(make_bptt_chunks(demo_sequence, bptt=4), 1):
    print(f"Chunk {i}: input={x.tolist()} target={y.tolist()}")

Hidden state dapat diteruskan antar-potongan, tetapi gradien diputus pada batas potongan. Karena itu, state mungkin membawa informasi lama, namun gradien tidak menelusuri tanpa batas ke masa lalu.

## 3. Vanishing dan exploding gradients
Backpropagation pada RNN melibatkan perkalian berulang dengan matriks bobot dan turunan aktivasi. Gradien bisa **vanish** (mengecil mendekati nol) atau **explode** (membesar sangat cepat). Vanishing gradients membuat pembelajaran dependensi jangka panjang sulit; exploding gradients dapat membuat pembaruan parameter tidak stabil.

In [ ]:
steps = 30
for label, factor in [("vanishing-like", 0.5), ("exploding-like", 1.5)]:
    values = [1.0]
    for _ in range(steps):
        values.append(values[-1] * factor)
    print(label, "nilai setelah", steps, "langkah:", values[-1])
    print("nilai awal:", np.round(values[:6], 5))

Contoh perkalian berulang di atas hanya mengisolasi efek numerik; ini bukan simulasi lengkap backpropagation RNN. *Gradient clipping* dapat membantu mengendalikan exploding gradients. LSTM dirancang untuk membantu mempertahankan informasi dan aliran gradien pada urutan panjang.

## 4. Long short-term memory (LSTM)
LSTM menambahkan *cell state* dan gerbang untuk mengatur aliran informasi:
- **Forget gate:** bagian mana dari state lama yang dipertahankan.
- **Input gate:** seberapa banyak informasi baru ditambahkan.
- **Output gate:** bagian state yang digunakan untuk membentuk hidden state.

Salah satu bentuk persamaan standar LSTM adalah:

\[
f_t=\sigma(W_f[x_t,h_{t-1}]+b_f),\quad i_t=\sigma(W_i[x_t,h_{t-1}]+b_i)
\]
\[
\tilde{c}_t=\tanh(W_c[x_t,h_{t-1}]+b_c),\quad
c_t=f_t\odot c_{t-1}+i_t\odot\tilde{c}_t
\]
\[
o_t=\sigma(W_o[x_t,h_{t-1}]+b_o),\quad
h_t=o_t\odot\tanh(c_t)
\]

\(\sigma\) adalah sigmoid dan \(\odot\) adalah perkalian element-wise. Ini ringkasan konseptual; implementasi berikut hanya demonstrasi satu langkah dengan bobot acak.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))

def lstm_step(x_t, h_prev, c_prev, params):
    combined = np.concatenate([x_t, h_prev])
    f = sigmoid(params["Wf"] @ combined + params["bf"])
    i = sigmoid(params["Wi"] @ combined + params["bi"])
    g = np.tanh(params["Wg"] @ combined + params["bg"])
    o = sigmoid(params["Wo"] @ combined + params["bo"])
    c = f * c_prev + i * g
    h = o * np.tanh(c)
    return h, c, {"forget": f, "input": i, "candidate": g, "output": o}

rng = np.random.default_rng(14)
input_size, hidden_size = 3, 4
def init_gate():
    return rng.normal(0, 0.2, size=(hidden_size, input_size + hidden_size))
params = {
    "Wf": init_gate(), "Wi": init_gate(), "Wg": init_gate(), "Wo": init_gate(),
    "bf": np.zeros(hidden_size), "bi": np.zeros(hidden_size),
    "bg": np.zeros(hidden_size), "bo": np.zeros(hidden_size)
}
h, c, gates = lstm_step(np.array([0.2, -0.1, 0.5]),
                         np.zeros(hidden_size), np.zeros(hidden_size), params)
print("hidden state:", np.round(h, 4))
print("cell state:", np.round(c, 4))
for name, value in gates.items():
    print(name, ":", np.round(value, 4))

## 5. Memilih karakter berikutnya
Model bahasa menghasilkan skor untuk setiap karakter. Softmax mengubah skor menjadi probabilitas. Saat menghasilkan teks, karakter dapat dipilih dari distribusi ini. *Temperature* rendah membuat distribusi lebih tajam; temperature tinggi membuatnya lebih rata.

Notebook ini tidak melatih model Shakespeare dan tidak menghasilkan teks yang dipelajari dari karya Shakespeare.

In [ ]:
def softmax(logits, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature harus lebih besar dari 0")
    scaled = np.asarray(logits, dtype=float) / temperature
    scaled -= np.max(scaled)
    exp_values = np.exp(scaled)
    return exp_values / exp_values.sum()

logits = np.array([2.0, 1.0, 0.2, -0.5])
for temperature in [0.5, 1.0, 2.0]:
    print(f"temperature={temperature}:", np.round(softmax(logits, temperature), 3))

## 6. Rangkuman dan refleksi
1. Character language modeling memprediksi karakter berikutnya dari karakter sebelumnya.
2. Truncated BPTT membatasi rentang waktu yang dilalui gradien.
3. Truncation menghemat komputasi, tetapi membatasi pembelajaran dependensi jangka panjang.
4. Vanilla RNN dapat mengalami vanishing atau exploding gradients.
5. LSTM menggunakan cell state dan gates untuk mengatur aliran informasi.

**Pertanyaan refleksi**
1. Apa perbedaan vocabulary berbasis kata dan berbasis karakter?
2. Mengapa BPTT penuh mahal untuk korpus panjang?
3. Apa kompromi utama truncated BPTT?
4. Bagaimana vanishing gradients memengaruhi dependensi jangka panjang?
5. Apa fungsi forget gate, input gate, dan output gate?

**Batasan:** data dan bobot di notebook ini adalah demonstrasi. Untuk mereplikasi eksperimen buku, diperlukan dataset Shakespeare dan implementasi training yang sesuai. 